In [89]:

# INITIAL WARNING & SAFE DEFAULTS

import warnings

# Hide unnecessary warnings
warnings.filterwarnings("ignore")

# Safe variable initialization to avoid NameError
articles = []
summaries = []
report = ""
insight = ""
positive = 0
neutral = 0
negative = 0

# Information message for the user
st.warning(
    "Enter a company or sector in the sidebar and click **Generate Report** "
    "to start the AI equity research analysis."
)

2026-03-13 05:52:39.607 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:39.612 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:39.617 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [90]:
# SUPPRESS STREAMLIT WARNINGS

import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("streamlit").setLevel(logging.ERROR)

In [91]:
pip install streamlit groq newsapi-python textblob pandas plotly

In [92]:
import streamlit as st
from groq import Groq
from newsapi import NewsApiClient
import pandas as pd
from textblob import TextBlob
import plotly.express as px

In [93]:
st.set_page_config(
    page_title="AI Equity Research Terminal",
    layout="wide"
)

2026-03-13 05:52:45.489 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [94]:
st.markdown("""
<style>

.stApp {
    background-color: #0e1117;
    color: white;
}

section[data-testid="stSidebar"] {
    background-color: #111827;
}

h1, h2, h3 {
    color: #facc15;
}

div[data-testid="stMetric"] {
    background-color: #1f2937;
    padding: 15px;
    border-radius: 10px;
}

</style>
""", unsafe_allow_html=True)

2026-03-13 05:52:45.497 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.500 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.500 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [95]:
GROQ_API_KEY = "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"
NEWS_API_KEY = "d7a63109ca0e4418b26d38f18079370f"

client = Groq(api_key=GROQ_API_KEY)
newsapi = NewsApiClient(api_key=NEWS_API_KEY)

In [96]:
st.sidebar.header("Research Settings")

query = st.sidebar.text_input("Enter Company or Sector")

generate = st.sidebar.button("Generate Report")

2026-03-13 05:52:45.580 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.583 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.585 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.586 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.588 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.589 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.590 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.591 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [97]:
def get_news(query):

    articles = newsapi.get_everything(
        q=query,
        language="en",
        page_size=5
    )

    return articles["articles"]

In [98]:
def summarize_articles(articles):

    summaries = []

    for article in articles:

        prompt = f"Summarize this news article:\n{article['title']} {article['description']}"

        response = client.chat.completions.create(
            model="llama3-70b-8192",
            messages=[{"role": "user", "content": prompt}]
        )

        summary = response.choices[0].message.content
        summaries.append(summary)

    return summaries

In [99]:
def generate_report(query, summaries):

    prompt = f"""
    Generate an equity research report about {query}
    using the following news summaries:

    {summaries}
    """

    response = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [100]:
if generate and query:

    with st.spinner("Generating AI Research..."):

        articles = get_news(query)

        summaries = summarize_articles(articles)

        report = generate_report(query, summaries)

In [101]:
    positive = 0
    neutral = 0
    negative = 0

    for summary in summaries:

        polarity = TextBlob(summary).sentiment.polarity

        if polarity > 0:
            positive += 1
        elif polarity < 0:
            negative += 1
        else:
            neutral += 1

In [102]:
    sentiment_df = pd.DataFrame({
        "Sentiment": ["Positive","Neutral","Negative"],
        "Count": [positive,neutral,negative]
    })

    fig_sentiment = px.bar(
        sentiment_df,
        x="Sentiment",
        y="Count",
        color="Sentiment",
        title="Market Sentiment Analysis"
    )

In [103]:
    companies = []

    for summary in summaries:
        words = summary.split()

        for word in words:
            if word.istitle():
                companies.append(word)

    company_counts = Counter(companies)

    top_companies = dict(company_counts.most_common(10))

In [104]:
    company_df = pd.DataFrame({
        "Company": list(top_companies.keys()),
        "Mentions": list(top_companies.values())
    })

    fig_company = px.bar(
        company_df,
        x="Company",
        y="Mentions",
        title="Top Companies Mentioned"
    )

In [105]:
    insight_prompt = f"""
    Based on these news summaries about {query},
    give a short market insight.

    {summaries}
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": insight_prompt}]
    )

    insight = response.choices[0].message.content

In [106]:
    tab1, tab2, tab3 = st.tabs([
        "📄 Research Report",
        "📰 News Feed",
        "📊 Market Insights"
    ])

2026-03-13 05:52:45.912 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.913 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.914 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.915 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [107]:
    with tab1:

        st.subheader("AI Equity Research Report")

        st.write(report)

        st.download_button(
            "Download Report",
            report,
            "research_report.txt"
        )

2026-03-13 05:52:45.924 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.926 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.927 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.928 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.929 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.930 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.931 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.931 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [108]:
    with tab2:

        for article in articles:

            st.markdown(f"### {article['title']}")

            if article["description"]:
                st.write(article["description"])

            st.markdown(f"[Read Full Article]({article['url']})")

            st.markdown("---")

In [109]:
    with tab3:

        st.subheader("Market Insights")

        col1, col2, col3 = st.columns(3)

        col1.metric("Positive", positive)
        col2.metric("Neutral", neutral)
        col3.metric("Negative", negative)

        st.plotly_chart(fig_sentiment)

        st.plotly_chart(fig_company)

        st.subheader("AI Insight")

        st.info(insight)

2026-03-13 05:52:45.952 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.953 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.954 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.955 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.955 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.956 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.957 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-13 05:52:45.958 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar